In [1]:
%pip install cdsapi
%pip install xarray netcdf4 matplotlib
%pip install h5netcdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached xarray-2026.4.0-py3-none-any.whl.metadata (12 kB)
  Using cached netcdf4-1.7.4-cp311-abi3-win_amd64.whl.metadata (2.1 kB)
  Using cached cftime-1.6.5-cp313-cp313-win_amd64.whl.metadata (8.8 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached xarray-2026.4.0-py3-none-any.whl (1.4 MB)
Using cached netcdf4-1.7.4-cp311-abi3-win_amd64.whl (21.3 MB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 1.7 MB/s eta 0:00:05
   ----- ---------------------------------- 1.0/8.2 MB 1.6 MB/s eta 0:00:05
   ------ --------------------------------- 1.3/8.2 MB 1.8 MB/s eta 0:00:04
   ------- -------------------------------- 1.6/8.2 MB 1.5 MB/s eta 0:00:05
   ----------- ---------------------------- 2.4/8.2 MB 1.9 MB/s eta 0:00:04
   --------------


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached h5netcdf-1.8.1-py3-none-any.whl.metadata (15 kB)
Using cached h5netcdf-1.8.1-py3-none-any.whl (62 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install h5py

   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   --- ------------------------------------ 0.3/3.2 MB ? eta -:--:--
   ------------- -------------------------- 1.0/3.2 MB 4.0 MB/s eta 0:00:01
   ----------------------- ---------------- 1.8/3.2 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 2.9/3.2 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 3.2/3.2 MB 3.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
# Configuramos el mensajero para que vaya al portal Climático normal
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write('url: https://cds.climate.copernicus.eu/api\n')
    f.write('key: b1c8ed63-d58a-4ca0-b6a8-0068ce339221')


In [8]:
import cdsapi
import os

# Configuración de la petición
c = cdsapi.Client()

dataset = "reanalysis-era5-land"
request = {
    "variable": ["total_precipitation"],
    "year": "2025",
    "month": "03",
    "day": [f"{i:02d}" for i in range(1, 32)],
    "time": [f"{i:02d}:00" for i in range(24)],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [43.51, -4.85, 42.76, -3.15]
}

target_nc = "precipitaciones_raw.nc"

# Ejecución de la descarga
if os.path.exists(target_nc):
    os.remove(target_nc) # Limpiamos para evitar conflictos

print("Conectando con Copernicus... Esto puede tardar unos minutos.")
c.retrieve(dataset, request).download(target_nc)
print(f"Archivo descargado: {target_nc} ({os.path.getsize(target_nc) / 1024:.2f} KB)")

Conectando con Copernicus... Esto puede tardar unos minutos.


2026-05-03 18:40:12,693 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-05-03 18:40:12,695 INFO Request ID is b99aea2f-c220-452d-a9e7-616197b4c1a5
2026-05-03 18:40:12,826 INFO status has been updated to accepted
2026-05-03 18:40:26,792 INFO status has been updated to running
2026-05-03 18:40:34,532 INFO status has been updated to successful
                                                                                      

Archivo descargado: precipitaciones_raw.nc (260.04 KB)


In [ ]:
import xarray as xr
import pandas as pd

input_file = "precipitaciones_raw.nc"
output_csv = "copernicus_precipitaciones_hist.csv"

try:
    # Abrimos el archivo
    ds = xr.open_dataset(input_file, engine='h5netcdf')
    
    # Convertimos todo el contenido a un DataFrame de una sola vez
    df = ds.to_dataframe().reset_index()
    
    # Buscamos la columna de precipitación para pasarla a mm
    # (Buscamos 'tp' o cualquier columna que contenga 'precip')
    for col in df.columns:
        if 'tp' in col or 'precip' in col.lower():
            df[col] = df[col] * 1000
            print(f"Columna '{col}' convertida a mm.")

    # Guardar el CSV con todas las columnas que traiga el archivo
    df.to_csv(output_csv, index=False)
    
    print(f"--- ¡CSV GENERADO! ---")
    print(f"Ruta: {output_csv}")
    display(df.head())

except Exception as e:
    print(f"Error: {e}")

Columna 'tp' convertida a mm.
--- ¡CSV GENERADO! ---
Ruta: datos_lluvia_tfm.csv


,valid_time,latitude,longitude,tp,number,expver
0,2025-03-01,43.5,-4.8,NaN,0,0001
1,2025-03-01,43.5,-4.7,NaN,0,0001
2,2025-03-01,43.5,-4.6,NaN,0,0001
3,2025-03-01,43.5,-4.5,NaN,0,0001
4,2025-03-01,43.5,-4.4,NaN,0,0001
